# Analyse RMSE — Osiris vs Fine Tuning

Ce notebook construit deux tableaux de **RMSE** :
1. **RMSE par champ de test** : chaque ligne = un champ (sonde regroupée), chaque colonne = un modèle × profondeur.
2. **RMSE par champ de validation** : chaque ligne = un champ de validation (`val_site`), chaque colonne = un modèle × profondeur.

Les valeurs **élevées** (au-delà d'un seuil) sont automatiquement mises **en gras**.

> - `model = lstm` → **Osiris** | `model = lstm_fine_tuned` → **Fine Tuning**
> - `type = OSIRIS_<site>_<probe>` → probe regroupée en champ = `OSIRIS_<site>` (extraction via `val_site`)
> - `metric_name = rmse_osiris` → RMSE (m³/m³), valeur = 7 horizons séparés par virgules
> - Profondeurs : 0.1, 0.2, 0.3, 0.4, 0.5 m

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
import ipywidgets as widgets

# ── Chemins à adapter si besoin ────────────────────────────────
RESULTS_CSV = "/home/theodore/Téléchargements/results.csv"
MODEL_LABELS = {"lstm": "Osiris", "lstm_fine_tuned": "Fine Tuning"}

In [ ]:
# ── Chargement ─────────────────────────────────────────────────
df = pd.read_csv(RESULTS_CSV, low_memory=False)
print(f"{len(df):,} lignes chargées.")

# Valeurs RMSE = chaîne d'horizons séparés par virgules
df["_horizons"] = df["value"].apply(
    lambda s: [float(x) for x in s.split(",")] if isinstance(s, str) and s != "" else [])
df["rmse_mean"] = df["_horizons"].apply(lambda v: float(np.mean(v)) if v else np.nan)

# Lignes RMSE OSIRIS uniquement
rmse = df[(df["metric_name"] == "rmse_osiris") & df["rmse_mean"].notna()].copy()

# Extraction du champ (sonde supprimée) via les val_site connus
site_ids = sorted(rmse["val_site"].unique())

def extract_champ(typ):
    suffix = typ.replace("OSIRIS_", "")
    for sid in sorted(site_ids, key=len, reverse=True):
        if suffix == sid or suffix.startswith(sid + "_"):
            return "OSIRIS_" + sid
    return typ

rmse["champ"] = rmse["type"].apply(extract_champ)

# Profondeurs et modèles
depths = sorted(rmse["depth"].unique())
models = [m for m in ["lstm", "lstm_fine_tuned"] if m in rmse["model"].unique()]

print(f"Champs : {rmse['champ'].nunique()} ({sorted(rmse['champ'].unique())})")
print(f"Profondeurs : {depths}")
print(f"Modèles : {[MODEL_LABELS.get(m, m) for m in models]}")
print(f"Lignes RMSE : {len(rmse):,}")

In [ ]:
# ── Paramètres ────────────────────────────────────────────────
THRESHOLD_MODE = "quantile"   # ou "mean_std"
THRESHOLD_QUANTILE = 0.75
THRESHOLD_N_STD = 1.0
PRECISION = 3

In [ ]:
def build_multi_pivot(index_col, threshold_mode=THRESHOLD_MODE):
    """Pivot multi-index colonnes (modèle × profondeur) x champ/val_site."""
    # Agréger sur les sondes + horizons → (index_col, model, depth) → rmse
    grp = (rmse.groupby([index_col, "model", "depth"])["rmse_mean"]
           .mean().reset_index())

    # Pivot : lignes = index_col, colonnes = (model, depth)
    pivot = grp.pivot_table(index=index_col, columns=["model", "depth"],
                            values="rmse_mean", aggfunc="first")

    # Renommer modèles
    pivot.columns = pd.MultiIndex.from_tuples(
        [(MODEL_LABELS.get(m, m), f"{d}m") for m, d in pivot.columns],
        names=["Modèle", "Profondeur"])

    return pivot.sort_index()


def highlight_high(row, thr):
    """Mets en gras les valeurs >= thr."""
    return ["font-weight: bold" if pd.notna(v) and v >= thr else "" for v in row]


def style_multi_pivot(pivot):
    """Style le pivot multi-index : gras + formatage."""
    vals = pivot.to_numpy(dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        return pivot.style, float("nan")

    if THRESHOLD_MODE == "mean_std":
        thr = float(np.mean(vals) + THRESHOLD_N_STD * np.std(vals))
    else:
        thr = float(np.quantile(vals, THRESHOLD_QUANTILE))

    styled = pivot.style.apply(lambda r: highlight_high(r, thr), axis=1)
    styled = styled.format(f"{{:.{PRECISION}f}}", na_rep="\u2013")
    return styled, thr

## 1. RMSE par champ de test

Chaque ligne = un **champ** (`OSIRIS_<site>`), regroupant toutes les sondes du champ. La RMSE est la moyenne sur les sondes, les plis LOO et les horizons de chaque (profondeur × modèle). Les valeurs **hautes** (RMSE ≥ seuil) sont en gras.

In [ ]:
pivot_test = build_multi_pivot("champ")
table_test, thr_test = style_multi_pivot(pivot_test)
print(f"Seuil de mise en gras : RMSE >= {thr_test:.3f}")
display(table_test)

## 2. RMSE par champ de validation

Chaque ligne = un **champ de validation** (`val_site`, leave-one-field-out). La RMSE est la moyenne sur tous les champs testés, les sondes, les plis et les horizons de chaque (profondeur × modèle).

In [ ]:
pivot_val = build_multi_pivot("val_site")
table_val, thr_val = style_multi_pivot(pivot_val)
print(f"Seuil de mise en gras : RMSE >= {thr_val:.3f}")
display(table_val)

## 3. (Option) Vue par profondeur seule

In [ ]:
# Pivot simple : (champ x model) pour une profondeur donnée
def show_by_depth(depth):
    sub = rmse[rmse["depth"] == depth]
    grp = sub.groupby(["champ", "model"])["rmse_mean"].mean().reset_index()
    pv = grp.pivot(index="champ", columns="model", values="rmse_mean")
    pv = pv.rename(columns=MODEL_LABELS)
    pv = pv[[c for c in ["Osiris", "Fine Tuning"] if c in pv.columns]].sort_index()

    vals = pv.to_numpy(dtype=float).ravel()
    vals = vals[~np.isnan(vals)]
    thr = float(np.quantile(vals, THRESHOLD_QUANTILE)) if len(vals) else 0

    styled = pv.style.apply(lambda r: highlight_high(r, thr), axis=1)
    styled = styled.format(f"{{:.{PRECISION}f}}", na_rep="\u2013")
    print(f"Profondeur {depth}m — seuil gras ≥ {thr:.3f}")
    display(styled)

widgets.interact(show_by_depth, depth=depths)